# Mission 2 Train

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# 파일 전체 디스크립션
# -----------------------------------------------------------------------------
# 이 코드는 위성/항공 이미지에서 폴리라인(예: 굴뚝 라인) 정보를 이용해
# 각 라인(물체)의 실제 높이(연속값)를 회귀하는 파이프라인입니다.
#
# 1) Backbone: ConvNeXt-Tiny로 이미지에서 feature map 추출
# 2) ROIAlign: 각 폴리라인의 두 끝점(x1,y1,x2,y2)로 정의된 선분을
#    - 정사각형 ROI(한 변 = max(|dx|,|dy|) * ROI_PAD_NORM)로 만들고
#    - 이미지 밖으로 나가지 않도록 중심(cx,cy)을 클램프(이때의 이동량 d_center도 기록)
#    - feature map에서 ROIAlign(7x7)으로 49개 토큰 생성(+학습 가능한 위치 임베딩)
# 3) Query(좌표 임베딩): 7D 기하 피처(dx, dy, sinθ, cosθ, cx, cy, side)에
#    2D(center shift: d_center_x, d_center_y)를 더해 9D → MLP로 hidden 차원으로 투영
# 4) Cross-Attention: (Query ↔ ROI 토큰들) 교차 어텐션 블록을 여러 층 통과
# 5) Head: 최종적으로 라인별 높이(스칼라) 회귀
#
# 데이터/로더:
#  - VIA 형식 유사 JSON(각 이미지에 polyline 및 'chi_height_m' 존재)을 읽어
#    (이미지, 정규화 좌표[0..1], 높이)로 구성
#  - 학습 시 좌우반전/밝기 증강
#  - 배치 내 라인 수를 맞추기 위해 0-padding + boolean mask 사용
#
# 손실/지표:
#  - masked MSE(마스크된 유효 라인만 평균)
#  - 평가 시 RMSE/MAE
#
# 오버샘플링:
#  - 이미지 단위 대표 높이(기본 max, 옵션 mean)
#  - log1p 변환 후 값 기반 균등 구간으로 binning
#  - 각 bin 역빈도 가중치로 WeightedRandomSampler 구성
#
# 옵티마이저:
#  - AdamW, LayerNorm/ bias / pos-embed(roi_pos)는 weight decay 제외
#  - ReduceLROnPlateau 스케줄러 + early stopping
# =============================================================================
import os, json, math, random
from typing import List, Tuple
from PIL import Image

import numpy as np  # ★ 추가
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler  # ★ Sampler 추가

from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torchvision.ops import roi_align   # ★ ROIAlign
from torchvision.transforms import functional as TF  # ★ 증강(F.hflip, adjust_brightness)용

# ==========
# 설정
# ==========
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 경로
TRAIN_IMG_DIR  = "/home/work/Seminar_Folder/A-team/형민_데이터/Data/Training/img/TS_KS"
TRAIN_LINE_DIR = "/home/work/Seminar_Folder/A-team/형민_데이터/Data/Training/label/TL_KS_LINE"
VAL_IMG_DIR    = "/home/work/Seminar_Folder/A-team/형민_데이터/Data/Validation/img/VS_KS"
VAL_LINE_DIR   = "/home/work/Seminar_Folder/A-team/형민_데이터/Data/Validation/label/VL_KS_LINE"

# 하이퍼파라미터
IMG_SIZE = 512
BATCH = 32
EPOCHS = 250
LR = 1e-5
WD = 1e-4
HIDDEN = 768
HEADS = 12
CROSS_LAYERS = 3      # CrossAttn 블록 층수
ROI_OUT = (7, 7)      # ROIAlign 출력 크기 (토큰=7x7=49)
ROI_PAD_NORM = 1.1    # ROI 스케일/패딩 팩터

# 증강 하이퍼파라미터 (학습에서만 사용)
AUG_P_HFLIP = 0.5
AUG_BRIGHTNESS_RANGE = (0.85, 1.15)  # 밝기 변화 범위(±15%)

# 오버샘플링 하이퍼파라미터
OVERSAMPLE_Q = 5          # 분위수 개수(기본 5분위)
OVERSAMPLE_STRATEGY = "max"  # 이미지 대표 높이: "max" 또는 "mean"
OVERSAMPLE_LOGSPLIT = True   # 분위수 분할 전 log1p 변환 여부

# ==========
# 유틸 & 데이터셋
# ==========
def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_float(x):
    try:
        return float(x)
    except:
        return None

def to_tensor_image(pil_img: Image.Image) -> torch.Tensor:
    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],
                             std =[0.229,0.224,0.225]),
    ])
    return tfm(pil_img)

def pad_and_stack(batch_imgs, batch_coords_list, batch_heights_list):
    # 배치 내 각 샘플의 라인 수(N)가 다를 수 있으므로,
    # 최대 라인 수(Nmax)에 맞춰 (coords, heights)를 뒤에 0으로 패딩.
    # 유효 라인 위치는 True, 패딩 위치는 False인 boolean mask도 함께 만든다.
    B = len(batch_imgs)
    Nmax = max([c.shape[0] for c in batch_coords_list]) if B>0 else 0
    coords_pad, heights_pad, mask = [], [], []
    for coords, h in zip(batch_coords_list, batch_heights_list):
        Ni = coords.shape[0]
        if Ni < Nmax:
            # 좌표 패딩: (x1,y1,x2,y2) → 0으로 채운다. 정규화 좌표라 0은 프레임 좌상단.
            pad_c = torch.zeros((Nmax-Ni, 4), dtype=coords.dtype)
            # 타깃 패딩: 높이 스칼라 → 0. 손실에서 mask로 제외되므로 값 자체는 의미 없음.
            pad_h = torch.zeros((Nmax-Ni,), dtype=h.dtype)
            coords = torch.cat([coords, pad_c], dim=0)
            h = torch.cat([h, pad_h], dim=0)
            # 유효 라인(True) + 패딩(False)
            m = torch.tensor([True]*Ni + [False]*(Nmax-Ni), dtype=torch.bool)
        else:
            m = torch.tensor([True]*Ni, dtype=torch.bool)
        coords_pad.append(coords); heights_pad.append(h); mask.append(m)
        
    # 이미지는 모두 동일 크기 텐서(3×H×W)라 바로 stack 가능
    imgs = torch.stack(batch_imgs, dim=0)
    coords = torch.stack(coords_pad, dim=0) # (B, Nmax, 4)
    heights = torch.stack(heights_pad, dim=0) # (B, Nmax)
    mask = torch.stack(mask, dim=0) # (B, Nmax)
    return imgs, coords, heights, mask

class LCTLineDataset(Dataset):
    """
    JSON으로부터 polyline과 'chi_height_m'(실측 높이)을 읽어
    이미지, 정규화 좌표[0..1], 높이를 제공.
    학습 시 좌우반전/밝기 증강을 적용하되, 좌표는 정규화 상태에서 x만 반전(1-x).
    """
    def __init__(self, img_dir: str, line_dir: str, include_prefix: str = None,
                 augment: bool = False, p_hflip: float = AUG_P_HFLIP,
                 brightness: Tuple[float, float] = AUG_BRIGHTNESS_RANGE):
        self.img_dir = img_dir
        self.line_dir = line_dir
        self.include_prefix = include_prefix  # 접두사 필터 (e.g., "K3" or "K3A", "K" -> 전체 코드)
        self.augment = augment
        self.p_hflip = p_hflip
        self.brightness = brightness
        self.items = []
        self._build()

    def _build(self):
        # line_dir 내 JSON을 순회하며 이미지 경로, 원본(W0,H0), polyline 및 높이 목록을 만든다.
        jfiles = [f for f in os.listdir(self.line_dir) if f.endswith(".json")]
        jfiles.sort()
        for jf in jfiles:
            jp = os.path.join(self.line_dir, jf)
            try:
                data = load_json(jp)
                key = next(iter(data.keys()))
                item = data[key]
                filename = item["filename"]

                # 접두사 필터
                if self.include_prefix is not None:
                    base = os.path.basename(filename)
                    if not base.startswith(self.include_prefix):
                        continue

                img_path = os.path.join(self.img_dir, filename)
                if not os.path.exists(img_path):
                    continue
                # 원본 이미지 크기. 없으면 PIL로 로딩해서 추출
                file_attr = item.get("file_attributes", {})
                W0 = safe_float(file_attr.get("img_width"))
                H0 = safe_float(file_attr.get("img_height"))
                if W0 is None or H0 is None:
                    with Image.open(img_path) as im0:
                        W0, H0 = im0.size
                polys, heights = [], []
                # regions 안에서 polyline만 수집. 시작점과 끝점(양 끝)을 선분으로 사용
                for r in item.get("regions", []):
                    sa = r.get("shape_attributes", {})
                    ra = r.get("region_attributes", {})
                    if sa.get("name") != "polyline": continue
                    xs = sa.get("all_points_x", []); ys = sa.get("all_points_y", [])
                    if len(xs) < 2 or len(ys) < 2:   continue
                    x1, y1 = float(xs[0]), float(ys[0])
                    x2, y2 = float(xs[-1]), float(ys[-1])
                    # 타깃 높이는 region_attributes['chi_height_m']
                    h = safe_float(ra.get("chi_height_m"))
                    if h is None: continue
                    polys.append((x1, y1, x2, y2)); heights.append(h)
                if len(polys) == 0: continue
                self.items.append({
                    "img_path": img_path,
                    "polys": polys,
                    "heights": heights,
                    "W0": float(W0),
                    "H0": float(H0),
                })
            except Exception:
                continue

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        img = Image.open(it["img_path"]).convert("RGB")
        W0, H0 = it["W0"], it["H0"]

        # 좌표 정규화: (x/W0, y/H0)로 0..1 구간으로 맞춤
        coords_list = []
        for (x1,y1,x2,y2) in it["polys"]:
            coords_list.append([x1/W0, y1/H0, x2/W0, y2/H0])
        coords = torch.tensor(coords_list, dtype=torch.float32)

        # ----- 학습용 증강 -----
        if self.augment:
            # 좌우 반전
            if random.random() < self.p_hflip:
                img = TF.hflip(img)
                # x 좌표 반전: x' = 1 - x  (정규화 좌표)
                coords[:, [0, 2]] = 1.0 - coords[:, [0, 2]]
            # 밝기 변화
            if self.brightness is not None and len(self.brightness) == 2:
                bmin, bmax = self.brightness
                if bmin > 0 and bmax > 0 and bmax >= bmin:
                    factor = random.uniform(bmin, bmax)
                    img = TF.adjust_brightness(img, factor)

        # Tensor 변환/정규화(크기 512x512 고정)
        img_t = to_tensor_image(img)
        heights = torch.tensor(it["heights"], dtype=torch.float32)
        return img_t, coords, heights

def lct_collate(batch):
    imgs, coords_list, heights_list = [], [], []
    for img_t, coords, heights in batch:
        imgs.append(img_t); coords_list.append(coords); heights_list.append(heights)
    return pad_and_stack(imgs, coords_list, heights_list)


# ==========
# 좌표 → 기하 피처 (7D)  (그대로 사용: c_geom 기반)
    # coords: (B, N, 4) = (x1,y1,x2,y2), 각 성분은 정규화 [0,1].
    # 산출: 7D 기하 피처 = [dx, dy, sinθ, cosθ, cx, cy, side]
    # side = max(|dx|,|dy|) : ROI 한 변의 기준이 되는 척도(정규화 길이).
# ==========
def build_coord_features(coords: torch.Tensor) -> torch.Tensor:
    # coords: (B,N,4) in [0,1]
    x1, y1, x2, y2 = coords.unbind(-1)
    dx, dy = x2 - x1, y2 - y1
    length01 = torch.sqrt(dx*dx + dy*dy + 1e-8) / math.sqrt(2.0)
    angle = torch.atan2(dy, dx)
    sin_th, cos_th = torch.sin(angle), torch.cos(angle)
    cx, cy = 0.5*(x1 + x2), 0.5*(y1 + y2)
    side = torch.maximum(torch.abs(dx), torch.abs(dy))
    feats = [dx, dy, sin_th, cos_th, cx, cy, side]  # 7D
    return torch.stack(feats, dim=-1) # (B, N, 7)


# ==========
# Cross-Attention 블록
# ==========
class CrossAttnBlock(nn.Module):
    # 표준 Transformer 블록 구조를 간소화하여 Query(라인별 1토큰) ↔ Key/Value(ROI 49토큰) 간 교차 어텐션을 수행.
    def __init__(self, hidden_dim, num_heads, attn_dropout=0.0, ffn_dropout=0.1):
        super().__init__()
        self.ln_q  = nn.LayerNorm(hidden_dim)
        self.ln_kv = nn.LayerNorm(hidden_dim)
        self.attn = nn.MultiheadAttention(embed_dim=hidden_dim,
                                          num_heads=num_heads,
                                          dropout=attn_dropout,
                                          batch_first=True)
        self.ffn = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim*4),
            nn.GELU(),
            nn.Dropout(ffn_dropout),
            nn.Linear(hidden_dim*4, hidden_dim)
        )
    def forward(self, q, kv, q_mask=None):
        # q: (K, 1, d), kv: (K, S, d)
        qn  = self.ln_q(q)
        kvn = self.ln_kv(kv)
        out,_ = self.attn(qn, kvn, kvn)   # (K,1,d)
        q = q + out
        q = q + self.ffn(q)
        if q_mask is not None:
            q = q * q_mask.unsqueeze(-1).unsqueeze(-1).float()
        return q

# ==========
# 모델: ConvNeXt → fmap, ROIAlign(7x7) → (라인별 49토큰) ↔ (좌표 쿼리) → 회귀
# ==========
class PolylineROIAlignModel(nn.Module):
    def __init__(self, hidden_dim=HIDDEN, num_heads=HEADS, num_layers=CROSS_LAYERS,
                 roi_out:Tuple[int,int]=ROI_OUT, fmap_hw=(16,16)):
        super().__init__()
        base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        self.backbone = base.features
        self.c_out = 768

        self.roi_proj = nn.Linear(self.c_out, hidden_dim)

        GEOM_IN = 9  # (7D c_geom + d_center 2D)
        self.coord_embed = nn.Sequential(
            nn.LayerNorm(GEOM_IN),   
            nn.Linear(GEOM_IN, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        S = roi_out[0]*roi_out[1]
        self.roi_pos = nn.Parameter(torch.zeros(1, S, hidden_dim)) # 위치 임베딩
        nn.init.trunc_normal_(self.roi_pos, std=0.02)

        self.blocks = nn.ModuleList([
            CrossAttnBlock(hidden_dim, num_heads, attn_dropout=0.0, ffn_dropout=0.1)
            for _ in range(num_layers)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.GELU(),
            nn.Linear(hidden_dim//2, 1) # 스칼라 높이 회귀
        ) 
        self.roi_out = roi_out
        self.fmap_hw = fmap_hw

    @staticmethod
    def _boxes_and_dcenter_shift_inside(coords_b: torch.Tensor,
                                        scale: float,
                                        img_size: int,
                                        eps: float = 1e-6) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        선분(x1,y1,x2,y2) → 정사각 ROI 박스로 변환하면서,
        ROI가 이미지 프레임(정규화 0..1) 밖으로 나가지 않게 중심(cx,cy)을 클램프.
        이때 발생한 중심 이동량 d_center=(cx'−cx, cy'−cy)를 Query 피처로 사용.

        Args:
            coords_b: (N,4) in [0,1]
            scale:    ROI 패딩 팩터(>1이면 여유를 더 줌), 예: 1.1
            img_size: 입력 이미지 한 변(px) — 여기선 512
        Returns:
            boxes_px: (N,4) in pixel (xmin,ymin,xmax,ymax), ROIAlign 입력
            d_center:(N,2) in norm  (Δcx, Δcy), 추가 기하 피처
        """
        x1, y1, x2, y2 = coords_b.unbind(-1)
        cx = 0.5*(x1+x2)
        cy = 0.5*(y1+y2)
        dx, dy = (x2-x1).abs(), (y2-y1).abs()
        side = torch.maximum(dx, dy) * scale
        side = side.clamp(min=eps, max=1.0)       # 너무 큰 ROI는 전체 이미지로 한정
        half = 0.5 * side

        # 이동 전(기하 중심)과 이동 후(풀링 중심)
        cxp = cx.clamp(half, 1.0 - half)
        cyp = cy.clamp(half, 1.0 - half)
        d_center = torch.stack([cxp - cx, cyp - cy], dim=-1)  # (N,2), 정규화 좌표계의 이동량

        xmin = (cxp - half) * img_size
        ymin = (cyp - half) * img_size
        xmax = (cxp + half) * img_size
        ymax = (cyp + half) * img_size
        boxes_px = torch.stack([xmin, ymin, xmax, ymax], dim=-1)  # (N,4)
        return boxes_px, d_center


    def forward(self, images: torch.Tensor, coords: torch.Tensor, query_mask: torch.Tensor):
        # images: (B,3,512,512), coords: (B,N,4 in [0,1]), query_mask: (B,N) bool
        B, _, H, W = images.shape
        assert H==IMG_SIZE and W==IMG_SIZE

        # ConvNeXt-Tiny feature map 추출
        fmap = self.backbone(images)                 # (B,C,Hf,Wf)
        _, C, Hf, Wf = fmap.shape
        # spatial_scale = (Hf / 512). ROIAlign 내부에서 px→feat좌표로 곱해짐.
        spatial_scale = Hf / float(IMG_SIZE)

        # 1) batched boxes & d_center (쿼리 순서와 정확히 동일하게 쌓음)
        boxes_list, dcenter_list, counts = [], [], []
        for b in range(B):
            valid = query_mask[b]
            if valid.any():
                cb = coords[b][valid]  # (Nv,4)
                boxes_px, d_center = self._boxes_and_dcenter_shift_inside(cb, ROI_PAD_NORM, IMG_SIZE)
                boxes_list.append(boxes_px)         # (Nv,4)
                dcenter_list.append(d_center)       # (Nv,2)  (정규화 좌표에서의 이동량)
                counts.append(boxes_px.size(0))
            else:
                boxes_list.append(torch.zeros((0,4), device=coords.device))
                dcenter_list.append(torch.zeros((0,2), device=coords.device))
                counts.append(0)
        total_boxes = sum(counts)
        if total_boxes == 0:
            Nmax = coords.size(1)
            return torch.zeros((B, Nmax), device=coords.device, dtype=images.dtype)

        # 2) ROIAlign
        pooled = roi_align(
            input=fmap, boxes=boxes_list, output_size=self.roi_out,
            spatial_scale=spatial_scale, sampling_ratio=-1, aligned=True
        )

        # 3) ROI tokens → hidden
        K = pooled.size(0)
        roi_h, roi_w = self.roi_out
        S = roi_h * roi_w
        roi_tokens = pooled.flatten(2).transpose(1,2)          # (K, S, C)
        roi_tokens = self.roi_proj(roi_tokens) + self.roi_pos  # (K, S, H)

        # 4) Query: (원래 7D 기하) + (d_center 2D) = 9D
        coord_feats = build_coord_features(coords)  # (B,N,9)  c_geom 기반
        qfeats_list = []
        dc_all = torch.cat(dcenter_list, dim=0) if dcenter_list else \
                 torch.zeros((0,2), device=coords.device)
        k_ptr = 0
        for b in range(B):
            m = query_mask[b]
            if m.any():
                base = coord_feats[b][m]                  # (Nv,7)
                dc   = dc_all[k_ptr:k_ptr+base.size(0)]   # (Nv,9)
                k_ptr += base.size(0)
                qfeats_list.append(torch.cat([base, dc], dim=-1))  # (Nv,9)
        q_feats = torch.cat(qfeats_list, dim=0)           # (K,9)
        q = self.coord_embed(q_feats)[:, None, :]         # (K,1,H)

        # 5) Cross-Attn stack
        for blk in self.blocks:
            q = blk(q, roi_tokens, q_mask=None)  # (K,1,H)

        # 6) 회귀 → (B,N) 복원
        pred_k = self.head(q).squeeze(-1).squeeze(-1)     # (K,)
        Nmax = coords.size(1)
        out = torch.zeros((B, Nmax), device=coords.device, dtype=pred_k.dtype)
        k_ptr = 0
        for b in range(B):
            m = query_mask[b]
            nv = int(m.sum().item())
            if nv > 0:
                out[b, m] = pred_k[k_ptr:k_ptr+nv]
                k_ptr += nv
        return out


# ==========
# 손실/지표
# ==========
def masked_mse(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor):
    # pred/tgt: (B,N), mask: (B,N) — True 위치만 평균 MSE.
    # 패딩된 자리(False)는 손실에서 제외되어, 배치 내 라인 수가 달라도 공정.
    diff = (pred - tgt)**2
    diff = diff[mask]
    return diff.mean() if diff.numel() > 0 else torch.tensor(0.0, device=pred.device)

@torch.no_grad()
def masked_metrics(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor):
    # 평가 시 RMSE/MAE를 유효 토큰만 대상으로 계산.
    p = pred[mask].float(); t = tgt[mask].float()
    if p.numel() == 0: return float("inf"), float("inf")
    rmse = float(torch.sqrt(((p - t)**2).mean()).item())
    mae  = float((p - t).abs().mean().item())
    return rmse, mae

# ==========
# Optimizer param grouping: Norm/ bias / pos-embed WD=0
# ==========
def build_param_groups(model: nn.Module, base_wd: float, base_lr: float):
    # AdamW에서 정규화 계층/바이어스/포지셔널 임베딩은 WD를 주지 않는 것이 일반적 관례.
    # LayerNorm은 통계/스케일 파라미터 성질상 WD를 주면 수렴을 방해할 수 있음

    # 1) 타입 기반으로 LayerNorm 파라미터 id 수집 (이름 의존 X)
    ln_param_ids = set()
    for m in model.modules():
        if isinstance(m, nn.LayerNorm):
            for p in m.parameters(recurse=False):
                ln_param_ids.add(id(p))

    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_ln   = id(p) in ln_param_ids          # 모든 LN 파라미터: WD=0
        is_bias = n.endswith(".bias")             # 모든 bias: WD=0
        is_pose = ("roi_pos" in n)                # pos-embed: WD=0
        if is_ln or is_bias or is_pose:
            no_decay.append(p)
        else:
            decay.append(p)

    return [
        {"params": decay,    "weight_decay": base_wd, "lr": base_lr},
        {"params": no_decay, "weight_decay": 0.0,     "lr": base_lr},
    ]

# ==========
# Oversampling 유틸
# ==========
def _representative_height(heights: List[float], strategy: str) -> float:
    if strategy == "mean":
        return float(np.mean(heights))
    # default: "max"
    return float(np.max(heights))

def make_weighted_sampler(dataset: LCTLineDataset,
                          n_bins: int = OVERSAMPLE_Q,
                          strategy: str = OVERSAMPLE_STRATEGY,
                          use_log_split: bool = OVERSAMPLE_LOGSPLIT) -> WeightedRandomSampler:
    """
    이미지 단위 Sampler:
      - 각 item(이미지)의 대표 높이(기본 max)를 추출
      - log1p 변환 후 min~max를 균등 간격으로 나눔 (equal-width binning)
      - 각 bin의 역빈도 가중치를 부여
    """
    # 이미지 대표 높이 목록
    rep = []
    for it in dataset.items:
        rep.append(_representative_height(it["heights"], strategy))
    rep = np.array(rep, dtype=np.float32)

    rep_for_split = np.log1p(rep) if use_log_split else rep

    # ★ 값 기반 균등 bin (equal-width)
    min_v, max_v = rep_for_split.min(), rep_for_split.max()
    q_edges = np.linspace(min_v, max_v, n_bins+1)

    # 각 이미지 bin index
    bin_idx = np.searchsorted(q_edges, rep_for_split, side="right") - 1
    bin_idx = np.clip(bin_idx, 0, n_bins-1)

    # bin별 count
    counts = np.bincount(bin_idx, minlength=n_bins).astype(np.float32)
    inv_freq = 1.0 / np.maximum(counts, 1e-6)

    # 샘플 weight
    weights = inv_freq[bin_idx]
    weights = torch.as_tensor(weights, dtype=torch.double)

    sampler = WeightedRandomSampler(weights, num_samples=len(dataset), replacement=True)

    print("[Oversampling] value bins edges:", q_edges)
    print("[Oversampling] bin counts:", counts)
    print("[Oversampling] inv_freq:", inv_freq)

    return sampler

# ==========
# 학습/검증 루프
# ==========
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss_weighted = 0.0   # ∑(배치 masked_mse × 유효 토큰 수)
    total_valid = 0             # ∑ 유효 토큰 수

    for imgs, coords, heights, mask in loader:
        imgs    = imgs.to(device, non_blocking=True)
        coords  = coords.to(device, non_blocking=True)
        heights = heights.to(device, non_blocking=True)
        mask    = mask.to(device, non_blocking=True)

        valid = int(mask.sum().item())
        if valid == 0:
            continue  # 유효 라인이 없는 배치 방어

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred = model(imgs, coords, mask)
            loss = masked_mse(pred, heights, mask)  # 배치 내 '유효 토큰 평균' MSE

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # 에포크 집계: 유효 토큰 수로 가중 평균
        total_loss_weighted += loss.item() * valid
        total_valid += valid

    return total_loss_weighted / max(1, total_valid)


@torch.no_grad()
def eval_epoch(model, loader):
    # 모든 배치의 유효 토큰에 대해 SSE/SAE를 누적 → 전역 MSE/RMSE/MAE 산출
    model.eval()
    se_sum, ae_sum, n_sum = 0.0, 0.0, 0
    for imgs, coords, heights, mask in loader:
        imgs   = imgs.to(device, non_blocking=True)
        coords = coords.to(device, non_blocking=True)
        heights= heights.to(device, non_blocking=True)
        mask   = mask.to(device, non_blocking=True)
        pred = model(imgs, coords, mask)
        diff = (pred - heights)[mask]
        se_sum += float((diff**2).sum().item())
        ae_sum += float(diff.abs().sum().item())
        n_sum  += int(mask.sum().item())
    if n_sum == 0: return float("inf"), float("inf"), float("inf") # 유효 표본이 없을 때는 무한대로 리턴하여 scheduler/early-stop이 인지하도록
    mse = se_sum / n_sum
    rmse = math.sqrt(mse)
    mae  = ae_sum / n_sum
    return mse, rmse, mae

# ==========
# 접두사별 개별 학습/검증 진입점
# ==========
def train_val_one_prefix(prefix: str):
    print(f"\n========== Training/Evaluating subset: {prefix} ==========")

    # 학습: 증강 + 오버샘플링
    train_ds = LCTLineDataset(TRAIN_IMG_DIR, TRAIN_LINE_DIR, include_prefix=prefix,
                              augment=True, p_hflip=AUG_P_HFLIP,
                              brightness=AUG_BRIGHTNESS_RANGE)
    # 검증: 원본 유지 (증강 X)
    val_ds   = LCTLineDataset(VAL_IMG_DIR,   VAL_LINE_DIR,   include_prefix=prefix,
                              augment=False)

    if len(train_ds) == 0 or len(val_ds) == 0:
        print(f"[{prefix}] Skipped: empty dataset (train={len(train_ds)}, val={len(val_ds)})")
        return

    # ★ Oversampling Sampler
    train_sampler = make_weighted_sampler(train_ds,
                                      n_bins=OVERSAMPLE_Q,
                                      strategy=OVERSAMPLE_STRATEGY,
                                      use_log_split=OVERSAMPLE_LOGSPLIT)


    # DataLoader 구성 (샘플러 사용 시 shuffle=False 필수)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=False,
                              sampler=train_sampler,
                              num_workers=2, pin_memory=True, collate_fn=lct_collate)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                              num_workers=2, pin_memory=True, collate_fn=lct_collate)

    # 모델은 접두사마다 새로 생성
    model = PolylineROIAlignModel(hidden_dim=HIDDEN, num_heads=HEADS,
                                  num_layers=CROSS_LAYERS, roi_out=ROI_OUT,
                                  fmap_hw=(16,16)).to(device)

    # Optimizer / Scheduler
    param_groups = build_param_groups(model, base_wd=WD, base_lr=LR)
    optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                           factor=0.5, patience=3)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_rmse, best_state = float("inf"), None
    patience, left = 30, 30 # 조기 종료 파라미터

    for epoch in range(1, EPOCHS+1):
        tr = train_one_epoch(model, train_loader, optimizer, scaler)
        va_mse, va_rmse, va_mae = eval_epoch(model, val_loader)
        scheduler.step(va_rmse)
        print(f"[{prefix}][{epoch:03d}/{EPOCHS}] train_loss={tr:.4f} | "
              f"val_loss={va_mse:.4f} | val_RMSE={va_rmse:.3f} | val_MAE={va_mae:.3f}")

        if va_rmse < best_rmse:
            best_rmse = va_rmse
            best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
            left = patience
        else:
            left -= 1
            if left <= 0:
                print(f"[{prefix}] Early stopping."); break

    os.makedirs("./checkpoints_roi_sq", exist_ok=True)
    save_path = f"./checkpoints_roi_sq/n_7D_10_7_roi_mv_ov_12d_zp_cd_un_{prefix}_polyline_roi_convnext_tiny_best.pth"
    torch.save(best_state if best_state is not None else model.state_dict(), save_path)
    print(f"[{prefix}] Saved best to: {save_path} (best val RMSE={best_rmse:.3f})")

# ==========
# 메인: 전체 모델 학습
# ==========
def main():
    for prefix in ["K"]:
        train_val_one_prefix(prefix)
if __name__ == "__main__":
    main()



========== Training/Evaluating subset: K ==========
[Oversampling] value bins edges: [3.7259343 4.1352224 4.544511  4.953799  5.3630867 5.772375 ]
[Oversampling] bin counts: [ 826. 1908. 3446. 1289.  583.]
[Oversampling] inv_freq: [0.00121065 0.00052411 0.00029019 0.0007758  0.00171527]


/tmp/ipykernel_125637/1969868220.py:590: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_125637/1969868220.py:510: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


[K][001/250] train_loss=17243.5490 | val_loss=11154.5782 | val_RMSE=105.615 | val_MAE=93.547
[K][002/250] train_loss=14838.5622 | val_loss=9353.3398 | val_RMSE=96.713 | val_MAE=83.364
[K][003/250] train_loss=12538.2771 | val_loss=7618.4225 | val_RMSE=87.284 | val_MAE=72.212
[K][004/250] train_loss=10975.3331 | val_loss=5978.6765 | val_RMSE=77.322 | val_MAE=60.342
[K][005/250] train_loss=8873.3922 | val_loss=4523.6063 | val_RMSE=67.258 | val_MAE=49.913
[K][006/250] train_loss=7571.2805 | val_loss=3356.7484 | val_RMSE=57.937 | val_MAE=42.154
[K][007/250] train_loss=6176.5882 | val_loss=2605.0093 | val_RMSE=51.039 | val_MAE=37.456
[K][008/250] train_loss=4857.8239 | val_loss=1715.3772 | val_RMSE=41.417 | val_MAE=26.027
[K][009/250] train_loss=3461.7515 | val_loss=1215.0693 | val_RMSE=34.858 | val_MAE=20.079
[K][010/250] train_loss=2693.7319 | val_loss=911.2730 | val_RMSE=30.187 | val_MAE=17.711
[K][011/250] train_loss=1834.9757 | val_loss=644.8963 | val_RMSE=25.395 | val_MAE=14.619
[K][01

# Mission 2 Validation

In [4]:
# ====== 추가: helper ======
import math, csv
import numpy as np
import torch

TOP_K = 10
OUT_DIR = "./error_examples"  # 결과 이미지 저장 폴더


def roi_center_shift_from_coord(coord_norm, scale=None, eps=1e-6):
    """
    coord_norm: [x1,y1,x2,y2] in [0,1] (폴리라인 양끝)
    정사각형 ROI(side = max(|dx|,|dy|)*scale)를 만들고,
    이미지 밖으로 나가지 않게 center를 [half, 1-half]로 clamp했을 때
    center가 이동했는지(d_center!=0)로 moved 판정.
    반환: moved(bool), (dcx,dcy), (x1',y1',x2',y2')  // 좌표는 [0,1]
    """
    if scale is None:
        scale = globals().get("ROI_PAD_NORM", 1.1)  # 모델 쪽과 맞춤

    x1, y1, x2, y2 = map(float, coord_norm)
    cx  = 0.5*(x1 + x2)
    cy  = 0.5*(y1 + y2)
    dx  = abs(x2 - x1)
    dy  = abs(y2 - y1)
    side = max(dx, dy) * float(scale)
    side = max(eps, min(1.0, side))     # [eps, 1.0]
    half = 0.5 * side

    # clamp된 center
    cxp = min(max(cx, half), 1.0 - half)
    cyp = min(max(cy, half), 1.0 - half)
    dcx, dcy = (cxp - cx), (cyp - cy)

    moved = (abs(dcx) > eps) or (abs(dcy) > eps)

    # 최종 정사각 박스 (정상화 좌표)
    x1p, y1p = (cxp - half), (cyp - half)
    x2p, y2p = (cxp + half), (cyp + half)
    # 수치오차 보정
    x1p = max(0.0, min(1.0, x1p)); y1p = max(0.0, min(1.0, y1p))
    x2p = max(0.0, min(1.0, x2p)); y2p = max(0.0, min(1.0, y2p))
    return moved, (dcx, dcy), (x1p, y1p, x2p, y2p)


# ====== 1) collect_errors: moved/d_center 추가 ======
@torch.no_grad()
def collect_errors(model, loader, dataset):
    model.eval()
    records = []
    img_ptr = 0

    for imgs, coords, heights, mask in loader:
        imgs   = imgs.to(device, non_blocking=True)
        coords = coords.to(device, non_blocking=True)
        heights= heights.to(device, non_blocking=True)
        mask   = mask.to(device, non_blocking=True)

        pred = model(imgs, coords, mask)

        B = imgs.size(0)
        for bi in range(B):
            ds_idx = img_ptr + bi
            item_meta = dataset.items[ds_idx]
            img_path  = item_meta["img_path"]

            m = mask[bi].bool()
            if not m.any():
                continue

            coords_b  = coords[bi, m].detach().cpu().numpy()  # (Nv,4) in [0,1]
            preds_b   = pred[bi, m].detach().cpu().numpy()
            targets_b = heights[bi, m].detach().cpu().numpy()

            for li in range(coords_b.shape[0]):
                p = float(preds_b[li]); t = float(targets_b[li])
                err = abs(p - t)
                c = coords_b[li]

                moved, dcenter, roi_sq = roi_center_shift_from_coord(c)  # ✅ 핵심
                dcx, dcy = dcenter

                records.append({
                    "img_idx": ds_idx,
                    "img_path": img_path,
                    "coord_norm": c,            # 원래 라인 끝점 기반
                    "roi_sq_norm": roi_sq,      # 만들어진 정사각형(참고용)
                    "d_center": dcenter,        # (dcx, dcy)
                    "moved": moved,             # ✅ 이동 여부
                    "pred": p,
                    "target": t,
                    "error": err,
                    "line_idx": li
                })
        img_ptr += B
    return records


# ====== 2) draw: moved 표시 ======
from PIL import Image, ImageDraw

def _draw_labelled_line(im, coord_norm, gt, pred, err, moved=False):
    draw = ImageDraw.Draw(im)
    W, H = im.size
    x1n, y1n, x2n, y2n = coord_norm
    x1, y1, x2, y2 = int(x1n * W), int(y1n * H), int(x2n * W), int(y2n * H)

    color = (255, 140, 0) if moved else (255, 0, 0)  # 주황: moved
    draw.line([(x1, y1), (x2, y2)], fill=color, width=3)
    text = f"GT:{gt:.2f}  Pred:{pred:.2f}  Err:{err:.2f}" + (" (moved)" if moved else "")
    # 간단히 위쪽 표기
    ty = max(0, y1 - 16)
    draw.text((max(0, min(W-150, x1)), ty), text, fill=(255, 255, 255))
    return im


# ====== 3) save_topk_error_images: moved 전달 + 파일명 suffix ======
def save_topk_error_images(prefix: str,
                           checkpoint_path: str,
                           out_dir: str = OUT_DIR,
                           top_k: int = TOP_K):
    print(f"\n========== Top-{top_k} Error Samples: {prefix} ==========")

    val_ds = LCTLineDataset(VAL_IMG_DIR, VAL_LINE_DIR, include_prefix=prefix, augment=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=2, pin_memory=True, collate_fn=lct_collate)

    model = PolylineROIAlignModel(hidden_dim=HIDDEN, num_heads=HEADS,
                                  num_layers=CROSS_LAYERS, roi_out=ROI_OUT,
                                  fmap_hw=(16,16)).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state)

    records = collect_errors(model, val_loader, val_ds)
    assert len(records) > 0, f"[{prefix}] no line records found in validation."

    records.sort(key=lambda r: r["error"], reverse=True)
    top_records = records[:top_k]

    save_dir = os.path.join(out_dir, prefix)
    os.makedirs(save_dir, exist_ok=True)

    saved_paths = []
    for rank, r in enumerate(top_records, 1):
        img_path = r["img_path"]
        coord_n  = r["coord_norm"]
        gt, pred, err = r["target"], r["pred"], r["error"]
        moved = bool(r.get("moved", False))

        im = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        im = _draw_labelled_line(im, coord_n, gt, pred, err, moved=moved)

        base = os.path.splitext(os.path.basename(img_path))[0]
        suffix = "_mv" if moved else ""
        save_name = f"{rank:02d}_{base}_line{r['line_idx']}{suffix}_gt{gt:.2f}_pr{pred:.2f}_err{err:.2f}.png"
        out_p = os.path.join(save_dir, save_name)
        im.save(out_p)
        saved_paths.append(out_p)

    print(f"[{prefix}] saved {len(saved_paths)} files to: {save_dir}")
    for p in saved_paths:
        print("  -", p)


# ====== 4) 리포트: moved subset RMSE/MAE + CSV moved 컬럼 ======
@torch.no_grad()
def print_validation_report(prefix: str,
                            checkpoint_path: str,
                            height_bin_edges=None,
                            out_dir: str = OUT_DIR,
                            save_detail_csv: bool = True,
                            save_report_txt: bool = True):
    val_ds = LCTLineDataset(VAL_IMG_DIR, VAL_LINE_DIR, include_prefix=prefix, augment=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=2, pin_memory=True, collate_fn=lct_collate)

    model = PolylineROIAlignModel(hidden_dim=HIDDEN, num_heads=HEADS,
                                  num_layers=CROSS_LAYERS, roi_out=ROI_OUT,
                                  fmap_hw=(16,16)).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state)

    records = collect_errors(model, val_loader, val_ds)
    assert len(records) > 0, f"[{prefix}] no line records found in validation."

    preds   = np.array([r["pred"]   for r in records], dtype=np.float64)
    targets = np.array([r["target"] for r in records], dtype=np.float64)
    abserr  = np.abs(preds - targets)
    sqerr   = (preds - targets) ** 2

    RMSE = float(np.sqrt(np.mean(sqerr)))
    MAE  = float(np.mean(abserr))
    p90, p95, p99 = [float(np.percentile(abserr, q)) for q in (90, 95, 99)]

    eps = 1e-6
    MAPE  = float(np.mean(abserr / (np.abs(targets) + eps)) * 100.0)
    SMAPE = float(np.mean(2.0 * abserr / (np.abs(preds) + np.abs(targets) + eps)) * 100.0)

    # ✅ moved subset 성능
    moved_mask = np.array([bool(r.get("moved", False)) for r in records], dtype=bool)
    if moved_mask.any():
        mv_rmse = float(np.sqrt(np.mean((preds[moved_mask] - targets[moved_mask])**2)))
        mv_mae  = float(np.mean(np.abs(preds[moved_mask] - targets[moved_mask])))
    else:
        mv_rmse, mv_mae = float("nan"), float("nan")

    # 구간별(기존 그대로)
    if height_bin_edges is None:
        max_t = float(np.max(targets))
        upper = max(50.0, math.ceil(max_t / 50.0) * 50.0)
        edges = np.arange(0.0, upper + 50.0, 50.0)
    else:
        edges = np.array(height_bin_edges, dtype=np.float64)
        if len(edges) < 2:
            raise ValueError("height_bin_edges must have at least two values.")
    edges_ext = np.concatenate([edges, [np.inf]])

    bin_rows = []
    for i in range(len(edges_ext) - 1):
        lo, hi = edges_ext[i], edges_ext[i + 1]
        sel = (targets >= lo) & (targets < hi)
        n = int(sel.sum())
        if n == 0:
            bin_rows.append([f"[{lo:g},{hi if np.isfinite(hi) else 'inf'}):", 0, np.nan, np.nan, np.nan, np.nan, np.nan])
            continue
        rmse_b = float(np.sqrt(np.mean(((preds[sel] - targets[sel]) ** 2))))
        mae_b  = float(np.mean(np.abs(preds[sel] - targets[sel])))
        p95_b  = float(np.percentile(np.abs(preds[sel] - targets[sel]), 95))
        mape_b = float(np.mean(np.abs(preds[sel] - targets[sel]) / (np.abs(targets[sel]) + eps)) * 100.0)
        smape_b= float(np.mean(2.0 * np.abs(preds[sel] - targets[sel]) / (np.abs(preds[sel]) + np.abs(targets[sel]) + eps)) * 100.0)
        bin_rows.append([f"[{lo:g},{hi if np.isfinite(hi) else 'inf'}):", n, rmse_b, mae_b, p95_b, mape_b, smape_b])

    # 출력
    print(f"\n========== Validation Report: {prefix} ==========")
    print(f"Count: {len(records)}")
    print(f"Overall - RMSE: {RMSE:.3f} | MAE: {MAE:.3f} | |err| P90/P95/P99: {p90:.2f} / {p95:.2f} / {p99:.2f} | "
          f"MAPE: {MAPE:.2f}% | SMAPE: {SMAPE:.2f}%")
    print(f"[박스 이동 굴뚝] Count={int(moved_mask.sum())} | RMSE={mv_rmse:.3f} | MAE={mv_mae:.3f}")

    header = ["Height Bin", "Count", "RMSE", "MAE", "P95 |err|", "MAPE %", "SMAPE %"]
    w = [12, 7, 8, 8, 10, 9, 10]
    print("-" * (sum(w) + len(w) * 3))
    print("{:<12} {:>7} {:>8} {:>8} {:>10} {:>9} {:>10}".format(*header))
    print("-" * (sum(w) + len(w) * 3))
    for row in bin_rows:
        b, n, rm, ma, p95b, mape_b, smape_b = row
        print("{:<12} {:>7} {:>8} {:>8} {:>10} {:>9} {:>10}".format(
            b, n,
            "nan" if np.isnan(rm) else f"{rm:.3f}",
            "nan" if np.isnan(ma) else f"{ma:.3f}",
            "nan" if np.isnan(p95b) else f"{p95b:.2f}",
            "nan" if np.isnan(mape_b) else f"{mape_b:.2f}",
            "nan" if np.isnan(smape_b) else f"{smape_b:.2f}",
        ))
    print("-" * (sum(w) + len(w) * 3))

    # 저장
    os.makedirs(out_dir, exist_ok=True)
    if save_detail_csv:
        csv_path = os.path.join(out_dir, f"{prefix}_val_details.csv")
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["img_path", "line_idx",
                             "gt", "pred", "abs_err", "mape_%", "smape_%",
                             "x1n", "y1n", "x2n", "y2n", "moved", "dcx", "dcy"])
            for r in records:
                gt = float(r["target"]); pr = float(r["pred"])
                ae = abs(pr - gt)
                mape_i  = 100.0 * ae / (abs(gt) + eps)
                smape_i = 100.0 * (2.0 * ae) / (abs(pr) + abs(gt) + eps)
                x1, y1, x2, y2 = r["coord_norm"]
                dcx, dcy = r.get("d_center", (0.0, 0.0))
                writer.writerow([r["img_path"], r["line_idx"], gt, pr, ae, mape_i, smape_i,
                                 x1, y1, x2, y2, bool(r.get("moved", False)), dcx, dcy])
        print(f"[{prefix}] saved detail CSV -> {csv_path}")

    if save_report_txt:
        txt_path = os.path.join(out_dir, f"{prefix}_val_report.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(f"Validation Report: {prefix}\n")
            f.write(f"Count: {len(records)}\n")
            f.write(f"Overall: RMSE={RMSE:.4f}, MAE={MAE:.4f}, |err| P90/P95/P99={p90:.3f}/{p95:.3f}/{p99:.3f}, "
                    f"MAPE={MAPE:.3f}%, SMAPE={SMAPE:.3f}%\n")
            f.write(f"Moved-only: Count={int(moved_mask.sum())}, RMSE={mv_rmse:.4f}, MAE={mv_mae:.4f}\n\n")
            f.write("{:<12} {:>7} {:>8} {:>8} {:>10} {:>9} {:>10}\n".format(*header))
            for row in bin_rows:
                b, n, rm, ma, p95b, mape_b, smape_b = row
                f.write("{:<12} {:>7} {:>8} {:>8} {:>10} {:>9} {:>10}\n".format(
                    b, n,
                    "nan" if np.isnan(rm) else f"{rm:.4f}",
                    "nan" if np.isnan(ma) else f"{ma:.4f}",
                    "nan" if np.isnan(p95b) else f"{p95b:.4f}",
                    "nan" if np.isnan(mape_b) else f"{mape_b:.4f}",
                    "nan" if np.isnan(smape_b) else f"{smape_b:.4f}",
                ))
        print(f"[{prefix}] saved summary TXT -> {txt_path}")


# ===== 사용 =====
if __name__ == "__main__":
    ckpt_dir = "/home/work/Seminar_Folder/A-team/형민_데이터/checkpoints_roi_sq"
    # 리포트 출력/저장 (구간은 0,50,100,...)
    print_validation_report("K", f"{ckpt_dir}/n_7D_10_7_roi_mv_ov_12d_zp_cd_un_K_polyline_roi_convnext_tiny_best.pth")



========== Validation Report: K ==========
Count: 1323
Overall - RMSE: 4.939 | MAE: 2.741 | |err| P90/P95/P99: 5.85 / 9.06 / 20.25 | MAPE: 2.90% | SMAPE: 2.87%
[박스 이동 굴뚝] Count=95 | RMSE=9.999 | MAE=5.416
-------------------------------------------------------------------------------------
Height Bin     Count     RMSE      MAE  P95 |err|    MAPE %    SMAPE %
-------------------------------------------------------------------------------------
[0,50.0):         94    5.815    2.772       7.79      6.06       5.57
[50,100.0):      540    4.152    2.451       7.62      3.32       3.28
[100,150.0):     497    5.088    2.792       9.08      2.28       2.32
[150,200.0):     113    5.529    3.355      12.64      2.00       2.01
[200,250.0):      53    7.340    3.691      13.48      1.63       1.67
[250,300.0):      26    4.831    3.058       5.67      1.11       1.12
[300,inf):         0      nan      nan        nan       nan        nan
------------------------------------------------------